# Развёртка макросов и AST

Команды компилируют отдельный фрагмент, но не выполняют его runtime-код. Compile-time макросы запускаются как обычно. Представление получено после вывода типов, до оптимизации.

In [ ]:
%%writefile notebook_macros.das
options gen2
module notebook_macros public
require daslib/ast
require daslib/ast_boost
require daslib/templates_boost
require daslib/macro_boost
require daslib/ast_verify
[call_macro(name="plus_one")]
class PlusOneMacro : AstCallMacro {
    def override visit(prog:ProgramPtr; mod:Module?; var expr:ExprCallMacro?):ExpressionPtr {
        macro_verify(length(expr.arguments)==1, prog, expr.at, "plus_one expects one argument")
        var result <- qmacro($e(clone_expression(expr.arguments[0])) + 1)
        verify_expression(result)
        return <- result
    }
}


In [ ]:
require ./notebook_macros.das
var value=40
var calls=0
def bump():int {calls+=1;return calls}
def sample(x:int):int {return plus_one(x)}

In [ ]:
%expand plus_one(value)

In [ ]:
%%ast
plus_one(value)

In [ ]:
%expand sample

In [ ]:
%%expand
def temporary(x:int):int {return plus_one(x)}

In [ ]:
%%ast
value=999
var temporary=plus_one(value)
print("THIS MUST NOT RUN")

In [ ]:
%expand bump()

Переменная `value` должна остаться 40, `calls` — 0. Функция `temporary` и локальные переменные из просмотренных фрагментов в сессию не добавляются.

In [ ]:
value

In [ ]:
calls

In [ ]:
%%ast
struct Point {x:int;y:float}

In [ ]:
%reset